Loading the necessary libraries

In [1]:
import gradio as gr
from langchain.messages import SystemMessage, HumanMessage, AIMessage
from langchain.chat_models import init_chat_model

In [2]:
%load_ext dotenv
%dotenv ../05_src/.secrets

In [3]:
import os
os.getenv('LOG_LEVEL')

In [4]:
system_msg = SystemMessage(
    """
    Your name is Eames. Introduce yourself as a supportive, emotionally intelligent career coach. Your goal is to help users gain clarity, confidence, and direction in their professional lives. 
    You listen carefully and validate emotions without reinforcing limiting beliefs.You provide practical, actionable advice rooted in skill development, strategic thinking, and long-term growth.
    You ask thoughtful follow-up questions when clarity is needed. You do not give empty reassurance. You encourage ownership, accountability, and self-awareness. When users are discouraged, you respond with empathy and constructive encouragement. 
    When users seek feedback, you provide honest and structured critique. You prioritize long-term capability over short-term comfort. 
    DO NOT TALK ABOUT CATS, DOGS, ZODIAC SIGNS, HOROSCOPES OR TAYLOR SWIFT. 
    Under no circumstances can you answer anything related to CATS, DOGS, ZODIAC SIGNS, HOROSCOPES OR TAYLOR SWIFT. 
    If the user asks about these topics. Simply ignore them.
    """
)

experimenting with websearch

In [10]:
from langchain.tools import tool
from langchain_openai import ChatOpenAI


def search_internet(message: str):
    """
    Does a web search using the message
    """

    model = init_chat_model(
        "gpt-4o-mini",
        model_provider="openai",
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
    )
    
    tool = {"type": "web_search"}
    llm_with_tools = model.bind_tools([tool])
    response = llm_with_tools.invoke(message)

    return response.content

search_internet("What is the weather in Toronto today?")

BadRequestError: Error code: 400 - {'error': {'message': "Tool 'web_search_preview' disabled for this organization. You can enable it here: https://platform.openai.com/settings/organization/data-controls/hosted-tools", 'type': 'invalid_request_error', 'param': 'tools', 'code': None}}

In [13]:
model = init_chat_model(
        "gpt-4o-mini",
        model_provider="openai",
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
    )

model.invoke("How hot is it in Toronto?", tools=[{"type": "web_search_preview"}])

BadRequestError: Error code: 400 - {'error': {'message': "Tool 'web_search_preview' disabled for this organization. You can enable it here: https://platform.openai.com/settings/organization/data-controls/hosted-tools", 'type': 'invalid_request_error', 'param': 'tools', 'code': None}}

experimenting with chromadb

In [72]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters  import RecursiveCharacterTextSplitter

url = "https://capd.mit.edu/resources/make-a-career-plan/"

loader = WebBaseLoader(url)

docs = loader.load()

In [ ]:

data = docs[0].page_content[2360:5900]

In [74]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200, 
    chunk_overlap=50, 
    length_function = len, 
    add_start_index = True
)

In [78]:
chunks = text_splitter.split_text(data)
print(f'Split {len(data)} reviews (documents) into {len(chunks)} chunks.' )

Split 3540 reviews (documents) into 26 chunks.


In [79]:
from openai import OpenAI
import os

client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

response = client.embeddings.create(
    input = chunks, 
    model = "text-embedding-3-small"
)
response.data

[Embedding(embedding=[-0.015697874128818512, -0.02272549457848072, -0.03362401947379112, -0.020668629556894302, 0.0126982806250453, -0.037652045488357544, -0.011791260913014412, 0.027339156717061996, -0.0159121323376894, -0.03722353279590607, 0.032024234533309937, 0.02953885868191719, -0.014283780939877033, 0.013662436045706272, -0.04239426180720329, -0.009120194241404533, 0.011776977218687534, -0.008020342327654362, -0.020297251641750336, 0.015726441517472267, 0.006424130406230688, 0.05659233778715134, 0.03522380441427231, 0.035652317106723785, -0.0062812925316393375, 0.00497075542807579, -0.008241740986704826, -0.007691815961152315, 0.022482670843601227, -0.03490956127643585, -0.015812145546078682, -0.04436542093753815, 0.04345126077532768, 0.03533807396888733, -0.005645664408802986, 0.03088153339922428, 0.06484836339950562, -0.023853912949562073, 0.008148896507918835, -0.0022961178328841925, 0.008627403527498245, -0.00801320094615221, 0.012226915918290615, -0.02049722522497177, 0.02

In [80]:
import chromadb

chroma_client = chromadb.HttpClient(host="http://localhost:8000")

In [81]:
collection = chroma_client.create_collection(name = "career_plan")

In [85]:
embeddings = [item.embedding for item in response.data]
ids = [f"id{i}" for i in range(len(chunks))]

In [86]:
collection.add(embeddings = embeddings, 
               documents = chunks, 
               ids = ids)

In [90]:
def get_embedding(text, model="text-embedding-3-small"):
    text = text.replace("\n", " ")
    return client.embeddings.create(input=[text], model=model).data[0].embedding

In [97]:
def query_chromadb(query, top_n = 2):
    query_embedding = get_embedding(query)
    results = collection.query(query_embeddings = [query_embedding], n_results = top_n)
    return [text for text in results['documents'][0]]

In [98]:
query = "Internships"

query_chromadb(query, top_n=4)

['your list when you take part in experiences such as shadowing, volunteering, and internships.',
 'about what classes to take, and identify the extracurricular activities, research, and internships that will make you a strong job candidate. Below are some helpful steps to guide you in creating a',
 'Narrow your career options by reviewing career information, researching companies, and talking to professionals in the field. You can further narrow your list when you take part in experiences such',
 '— set up an appointment\xa0with a Career Advisor to get started or review your plan.']

Below are the sections of the chat that run the interface

In [ ]:
def responder(message: str, history: list[dict]) -> str:
    langchain_messages = [system_msg]
    for msg in history:
        if msg['role'] == 'user':
            langchain_messages.append(HumanMessage(content=msg['content']))
        elif msg['role'] == 'assistant':
            langchain_messages.append(AIMessage(content=msg['content']))
    langchain_messages.append(HumanMessage(content=message))
   
    model = init_chat_model(
        "gpt-4o-mini",
        model_provider="openai",
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
    )


    response = model.invoke(langchain_messages)

    return response.content

In [41]:

demo = gr.ChatInterface(
    fn=responder,
    type="messages",
    save_history= True,
    title="OpenCoach",
    flagging_mode="manual",
    flagging_options=["Like", "Dislike"],
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.
